In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import pandas as pd
from utils import validate_confirmation_of_prior_product, validate_inputs_of_one
from dist_s1_enumerator import  get_rtc_s1_ts_metadata_from_mgrs_tiles, enumerate_dist_s1_products, enumerate_dist_s1_workflow_inputs
from tqdm import tqdm
from dist_s1.data_models.data_utils import get_track_number
import rasterio
import json
from concurrent.futures import ThreadPoolExecutor

/Users/cmarshak/miniforge3/envs/dist-s1-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Parameters

In [3]:
TXT_FILE_WITH_PRODUCTS = 'dist-s1_tile-ids_for_charlie_2026-01-21.txt'
S3_PREFIX = 's3://opera-int-rs-fwd/products/DIST_S1'
MGRS_TILE_ID = '37NEG'
DELTA_LOOKBACK_DAYS = (365, 730, 1095)
DELTA_WINDOW_DAYS = 60

TIME_BEGIN = '2024-01-01'
TIME_END = '2025-01-01'


In [4]:
out_dir = Path('out')
mgrs_tile_dir = out_dir / 'issues' / f'{MGRS_TILE_ID}'

# Format Products into a Table

In [5]:
with open('dist-s1_tile-ids_for_charlie_2026-01-21.txt') as f:
    prod_ids = f.read().splitlines()
prod_ids = list(map(lambda x: x.strip(), prod_ids))
prod_ids = list(filter(lambda x: x and ('missing' not in x), prod_ids))
prod_ids[:3]


['OPERA_L3_DIST-ALERT-S1_T33NUF_20240121T172038Z_20260116T230458Z_S1A_30_v0.1',
 'OPERA_L3_DIST-ALERT-S1_T33NUF_20240202T172038Z_20260117T005247Z_S1A_30_v0.1',
 'OPERA_L3_DIST-ALERT-S1_T33NUF_20240214T172037Z_20260117T010531Z_S1A_30_v0.1']

In [6]:
def get_rtc_track_number(path: str) -> int:
    with rasterio.open(path) as ds:
        tags = ds.tags()
    opera_rtc_id = tags["post_rtc_opera_ids"].split(",")[0]
    return get_track_number(opera_rtc_id)
    

def get_acquisition_datetime(dist_s1_id: str) -> pd.Timestamp:
    return pd.Timestamp(dist_s1_id.split('_')[4])


In [7]:
df_sds_prod = pd.DataFrame({'dist_s1_id': prod_ids})
df_sds_prod['status_s3_path'] = df_sds_prod.dist_s1_id.map(lambda x: f'{S3_PREFIX}/{x}/{x}_GEN-DIST-STATUS.tif')
df_sds_prod['mgrs_tile_id'] = df_sds_prod.dist_s1_id.map(lambda x: x.split('_')[3][1:])
df_sds_prod['acq_ts'] = df_sds_prod.dist_s1_id.map(get_acquisition_datetime)
df_sds_prod_one_tile = df_sds_prod[df_sds_prod.mgrs_tile_id == MGRS_TILE_ID]
df_sds_prod_one_tile = df_sds_prod_one_tile.sort_values(by='dist_s1_id')
df_sds_prod_one_tile.head()

,dist_s1_id,status_s3_path,mgrs_tile_id,acq_ts
83,OPERA_L3_DIST-ALERT-S1_T37NEG_20240104T031031Z...,s3://opera-int-rs-fwd/products/DIST_S1/OPERA_L...,37NEG,2024-01-04 03:10:31+00:00
84,OPERA_L3_DIST-ALERT-S1_T37NEG_20240111T030226Z...,s3://opera-int-rs-fwd/products/DIST_S1/OPERA_L...,37NEG,2024-01-11 03:02:26+00:00
85,OPERA_L3_DIST-ALERT-S1_T37NEG_20240116T031030Z...,s3://opera-int-rs-fwd/products/DIST_S1/OPERA_L...,37NEG,2024-01-16 03:10:30+00:00
86,OPERA_L3_DIST-ALERT-S1_T37NEG_20240123T030226Z...,s3://opera-int-rs-fwd/products/DIST_S1/OPERA_L...,37NEG,2024-01-23 03:02:26+00:00
87,OPERA_L3_DIST-ALERT-S1_T37NEG_20240128T031030Z...,s3://opera-int-rs-fwd/products/DIST_S1/OPERA_L...,37NEG,2024-01-28 03:10:30+00:00


In [8]:
with ThreadPoolExecutor(max_workers=10) as executor:
    track_numbers = list(tqdm(executor.map(get_rtc_track_number, df_sds_prod_one_tile.status_s3_path), total=df_sds_prod_one_tile.shape[0]))

df_sds_prod_one_tile['track_number'] = track_numbers
df_sds_prod_one_tile.head()


  0%|          | 0/59 [00:02<?, ?it/s]


RasterioIOError: Access Denied

In [110]:
if df_sds_prod_one_tile.empty:
    raise ValueError(f'No products found for tile {MGRS_TILE_ID}')


# Check Confirmation of prior product

In [ ]:
confirmation_validation = validate_confirmation_of_prior_product(df_sds_prod_one_tile.status_s3_path.tolist()[:])

In [114]:
confirmation_validation_incorrect = [v for v in confirmation_validation if not v[1]]
confirmation_validation_incorrect

[('OPERA_L3_DIST-ALERT-S1_T37NEG_20240123T030226Z_20260118T214506Z_S1A_30_v0.1_GEN-DIST-STATUS.tif',
  False)]

In [116]:
if confirmation_validation_incorrect:
    mgrs_tile_dir.mkdir(exist_ok=True, parents=True)
    with open(mgrs_tile_dir / 'confirmation_validation_incorrect.json', 'w') as f:
        json.dump(confirmation_validation_incorrect, f)


# Validate Inputs


In [117]:
MGRS_TILE_IDS = [MGRS_TILE_ID]
df_ts_for_one_mgrs = get_rtc_s1_ts_metadata_from_mgrs_tiles(
    MGRS_TILE_IDS,
)
df_ts_for_one_mgrs.head(5)

,opera_id,jpl_burst_id,acq_dt,acq_date_for_mgrs_pass,polarizations,track_number,pass_id,url_crosspol,url_copol,geometry,mgrs_tile_id,acq_group_id_within_mgrs_tile,track_token
0,OPERA_L2_RTC-S1_T006-011781-IW3_20160609T03014...,T006-011781-IW3,2016-06-09 03:01:42+00:00,2016-06-09,VV+VH,6,148,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,"POLYGON ((40.69726 6.15385, 39.99318 6.29763, ...",37NEG,0,6
1,OPERA_L2_RTC-S1_T006-011781-IW3_20161019T03014...,T006-011781-IW3,2016-10-19 03:01:48+00:00,2016-10-19,VV+VH,6,170,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,"POLYGON ((40.69944 6.1534, 39.99444 6.29737, 3...",37NEG,0,6
2,OPERA_L2_RTC-S1_T006-011781-IW3_20161112T03014...,T006-011781-IW3,2016-11-12 03:01:47+00:00,2016-11-12,VV+VH,6,174,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,"POLYGON ((40.6992 6.15368, 39.99419 6.29765, 3...",37NEG,0,6
3,OPERA_L2_RTC-S1_T006-011781-IW3_20161206T03014...,T006-011781-IW3,2016-12-06 03:01:47+00:00,2016-12-06,VV+VH,6,178,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,"POLYGON ((40.69844 6.15314, 39.99343 6.29712, ...",37NEG,0,6
4,OPERA_L2_RTC-S1_T006-011781-IW3_20161230T03014...,T006-011781-IW3,2016-12-30 03:01:46+00:00,2016-12-30,VV+VH,6,182,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,"POLYGON ((40.69831 6.15389, 39.9933 6.29785, 3...",37NEG,0,6


In [122]:
df_products = enumerate_dist_s1_products(df_ts_for_one_mgrs, 
                                         MGRS_TILE_IDS,
                                         delta_lookback_days=365, 
                                         delta_window_days=DELTA_LOOKBACK_DAYS,
                                         max_pre_imgs_per_burst=(4, 3, 3))
df_products.head()

Enumerate by MGRS tiles: 100%|██████████| 1/1 [00:03<00:00,  3.86s/it]


,opera_id,jpl_burst_id,acq_dt,acq_date_for_mgrs_pass,polarizations,track_number,pass_id,url_crosspol,url_copol,geometry,mgrs_tile_id,acq_group_id_within_mgrs_tile,track_token,input_category,product_id
0,OPERA_L2_RTC-S1_T006-011781-IW3_20221223T03022...,T006-011781-IW3,2022-12-23 03:02:22+00:00,2022-12-23,VV+VH,6,546,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,"POLYGON ((40.69942 6.15443, 39.99496 6.29828, ...",37NEG,0,6,pre,0
1,OPERA_L2_RTC-S1_T006-011782-IW3_20221223T03022...,T006-011782-IW3,2022-12-23 03:02:25+00:00,2022-12-23,VV+VH,6,546,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,"POLYGON ((40.66152 5.98841, 39.96018 6.13184, ...",37NEG,0,6,pre,0
2,OPERA_L2_RTC-S1_T006-011783-IW3_20221223T03022...,T006-011783-IW3,2022-12-23 03:02:28+00:00,2022-12-23,VV+VH,6,546,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,"POLYGON ((40.63114 5.82148, 39.92824 5.96545, ...",37NEG,0,6,pre,0
3,OPERA_L2_RTC-S1_T006-011784-IW3_20221223T03023...,T006-011784-IW3,2022-12-23 03:02:31+00:00,2022-12-23,VV+VH,6,546,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,"POLYGON ((40.60106 5.65383, 39.89206 5.79928, ...",37NEG,0,6,pre,0
4,OPERA_L2_RTC-S1_T006-011785-IW3_20221223T03023...,T006-011785-IW3,2022-12-23 03:02:33+00:00,2022-12-23,VV+VH,6,546,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,"POLYGON ((40.56805 5.48704, 39.85617 5.63331, ...",37NEG,0,6,pre,0


In [123]:
issues = [validate_inputs_of_one(p, df_products_all=df_products) for p in tqdm(df_sds_prod_one_tile.status_s3_path.tolist())]
issues[:3]

 15%|█▌        | 9/59 [00:01<00:05,  9.34it/s]/Users/cmarshak/bekaert-team/dist-s1-research/marshak/Za_check_pcm/utils.py:51: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_prod.dropna(subset=["time_window"]).groupby(["jpl_burst_id", "time_window"]).size().unstack(fill_value=0)
/Users/cmarshak/bekaert-team/dist-s1-research/marshak/Za_check_pcm/utils.py:51: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_prod.dropna(subset=["time_window"]).groupby(["jpl_burst_id", "time_window"]).size().unstack(fill_value=0)
 19%|█▊        | 11/59 [00:01<00:05,  8.88it/s]/Users/cmarshak/bekaert-team/dist-s1-research/marshak

[{'dist_s1_id': 'OPERA_L3_DIST-ALERT-S1_T37NEG_20240104T031031Z_20260118T203911Z_S1A_30_v0.1_GEN-DIST-STATUS',
  'issues': {},
  'hist': {}},
 {'dist_s1_id': 'OPERA_L3_DIST-ALERT-S1_T37NEG_20240111T030226Z_20260118T204514Z_S1A_30_v0.1_GEN-DIST-STATUS',
  'issues': {},
  'hist': {}},
 {'dist_s1_id': 'OPERA_L3_DIST-ALERT-S1_T37NEG_20240116T031030Z_20260118T212104Z_S1A_30_v0.1_GEN-DIST-STATUS',
  'issues': {},
  'hist': {}}]

In [124]:
issues_incorrect = [d for d in issues if d['issues']]
issues_incorrect[:3]


[{'dist_s1_id': 'OPERA_L3_DIST-ALERT-S1_T37NEG_20240304T031030Z_20260119T005111Z_S1A_30_v0.1_GEN-DIST-STATUS',
  'issues': {'issue_type': 'Pre RTC IDs found but not expected',
   'issue_value': ['OPERA_L2_RTC-S1_T079-168577-IW1_20221228T031026Z',
    'OPERA_L2_RTC-S1_T079-168577-IW2_20221228T031027Z',
    'OPERA_L2_RTC-S1_T079-168578-IW1_20221228T031029Z',
    'OPERA_L2_RTC-S1_T079-168578-IW2_20221228T031030Z',
    'OPERA_L2_RTC-S1_T079-168579-IW1_20221228T031032Z',
    'OPERA_L2_RTC-S1_T079-168579-IW2_20221228T031033Z',
    'OPERA_L2_RTC-S1_T079-168580-IW1_20221228T031035Z',
    'OPERA_L2_RTC-S1_T079-168580-IW2_20221228T031036Z',
    'OPERA_L2_RTC-S1_T079-168581-IW1_20221228T031038Z',
    'OPERA_L2_RTC-S1_T079-168581-IW2_20221228T031038Z',
    'OPERA_L2_RTC-S1_T079-168582-IW1_20221228T031040Z',
    'OPERA_L2_RTC-S1_T079-168582-IW2_20221228T031041Z',
    'OPERA_L2_RTC-S1_T079-168583-IW1_20221228T031043Z',
    'OPERA_L2_RTC-S1_T079-168583-IW2_20221228T031044Z',
    'OPERA_L2_RTC-S1_T079

In [127]:
if issues_incorrect:
    mgrs_tile_dir.mkdir(exist_ok=True, parents=True)
    with open(mgrs_tile_dir / 'issues_incorrect.json', 'w') as f:
        json.dump(issues_incorrect, f, indent=4)


# Check All Dates

In [21]:
DELTA_WINDOW_DAYS

60

In [16]:
# inputs = enumerate_dist_s1_workflow_inputs([MGRS_TILE_ID], track_numbers=None, start_acq_dt=TIME_BEGIN, stop_acq_dt=TIME_END, delta_lookback_days=DELTA_LOOKBACK_DAYS, delta_window_days=DELTA_WINDOW_DAYS)
inputs = enumerate_dist_s1_workflow_inputs(['47NRG'], track_numbers=None, start_acq_dt=TIME_BEGIN, stop_acq_dt=TIME_END, delta_lookback_days=DELTA_LOOKBACK_DAYS, delta_window_days=DELTA_WINDOW_DAYS)

Enumerate by MGRS tiles: 100%|██████████| 1/1 [00:04<00:00,  4.83s/it]


In [19]:
df_inputs_expected = pd.DataFrame(inputs)
df_inputs_expected.head()

,mgrs_tile_id,post_acq_date,track_number,post_acq_timestamp
0,47NRG,2024-01-04,91,2024-01-04 22:55:30+00:00
1,47NRG,2024-01-10,172,2024-01-10 11:27:02+00:00
2,47NRG,2024-01-16,91,2024-01-16 22:55:29+00:00
3,47NRG,2024-01-22,172,2024-01-22 11:27:01+00:00
4,47NRG,2024-01-28,91,2024-01-28 22:55:29+00:00


In [20]:
df_inputs_expected.head(30)

,mgrs_tile_id,post_acq_date,track_number,post_acq_timestamp
0,47NRG,2024-01-04,91,2024-01-04 22:55:30+00:00
1,47NRG,2024-01-10,172,2024-01-10 11:27:02+00:00
2,47NRG,2024-01-16,91,2024-01-16 22:55:29+00:00
3,47NRG,2024-01-22,172,2024-01-22 11:27:01+00:00
4,47NRG,2024-01-28,91,2024-01-28 22:55:29+00:00
5,47NRG,2024-02-02,164,2024-02-02 23:03:39+00:00
6,47NRG,2024-02-03,172,2024-02-03 11:27:01+00:00
7,47NRG,2024-02-09,91,2024-02-09 22:55:29+00:00
8,47NRG,2024-02-15,172,2024-02-15 11:27:01+00:00
9,47NRG,2024-02-21,91,2024-02-21 22:55:29+00:00


In [94]:
df_sds_prod_one_tile.sort_values(by='acq_ts', inplace=True)
df_sds_prod_one_tile['post_acq_date'] = df_sds_prod_one_tile.acq_ts.dt.date.astype(str)
df_sds_prod_one_tile.head()

,dist_s1_id,status_s3_path,mgrs_tile_id,acq_ts,track_number,post_acq_date
83,OPERA_L3_DIST-ALERT-S1_T37NEG_20240104T031031Z...,s3://opera-int-rs-fwd/products/DIST_S1/OPERA_L...,37NEG,2024-01-04 03:10:31+00:00,79,2024-01-04
84,OPERA_L3_DIST-ALERT-S1_T37NEG_20240111T030226Z...,s3://opera-int-rs-fwd/products/DIST_S1/OPERA_L...,37NEG,2024-01-11 03:02:26+00:00,6,2024-01-11
85,OPERA_L3_DIST-ALERT-S1_T37NEG_20240116T031030Z...,s3://opera-int-rs-fwd/products/DIST_S1/OPERA_L...,37NEG,2024-01-16 03:10:30+00:00,79,2024-01-16
86,OPERA_L3_DIST-ALERT-S1_T37NEG_20240123T030226Z...,s3://opera-int-rs-fwd/products/DIST_S1/OPERA_L...,37NEG,2024-01-23 03:02:26+00:00,6,2024-01-23
87,OPERA_L3_DIST-ALERT-S1_T37NEG_20240128T031030Z...,s3://opera-int-rs-fwd/products/DIST_S1/OPERA_L...,37NEG,2024-01-28 03:10:30+00:00,79,2024-01-28


In [95]:
expected_input_pairs = [(r['post_acq_date'], r['track_number']) for _, r in df_inputs_expected.iterrows()]
actual_input_pairs = [(r['post_acq_date'], r['track_number']) for _, r in df_sds_prod_one_tile.iterrows()]


In [96]:
expected_but_not_found = [p for p in expected_input_pairs if p not in actual_input_pairs]
found_but_not_expected = [p for p in actual_input_pairs if p not in expected_input_pairs]
expected_but_not_found, found_but_not_expected


([('2024-03-28', 79)], [])

In [ ]:
if expected_but_not_found:
    mgrs_tile_dir.mkdir(exist_ok=True, parents=True)
    with open(mgrs_tile_dir / 'expected_but_not_found_inputs.json', 'w') as f:
        json.dump(expected_but_not_found, f, indent=4)

if found_but_not_expected:
    mgrs_tile_dir.mkdir(exist_ok=True, parents=True)
    with open(mgrs_tile_dir / 'found_but_not_expected_inputs.json', 'w') as f:
        json.dump(found_but_not_expected, f, indent=4)